In [ ]:
## clean m2cai dataset

In [ ]:
## clean KCH



In [1]:
# GynSurg_Action

path = "data/Surge_Frames/GynSurg_Action/frames"

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 第一层文件夹是手术动作的标签，第二层文件夹是视频的名称，第三层是帧图片
path = "data/Surge_Frames/GynSurg_Action/frames"

data = []
global_idx = 0
videoid2num = dict()
curr_case_num = 0

# 1. 收集所有动作类别，建立action_name到数字标签的映射
action_names = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
action2id = {name: idx for idx, name in enumerate(action_names)}

for action_name in action_names:
    action_dir = os.path.join(path, action_name)
    if not os.path.isdir(action_dir):
        continue

    # 每个动作下每个视频文件夹的所有帧做为一个视频
    for video_name in sorted(os.listdir(action_dir)):
        video_dir = os.path.join(action_dir, video_name)
        if not os.path.isdir(video_dir):
            continue

        frame_list = [f for f in sorted(os.listdir(video_dir)) if f.lower().endswith(('.jpg', '.png'))]
        
        # ---- 任务1: 删除没有视频帧的文件夹 ----
        if len(frame_list) == 0:
            print(f"Deleting empty video folder: {video_dir}")
            os.rmdir(video_dir)
            continue
        
        # 映射CASE_ID: 唯一编号
        if video_name not in videoid2num:
            videoid2num[video_name] = curr_case_num
            curr_case_num += 1
        case_id = videoid2num[video_name]

        # 获取动作数字标签
        phase_gt = action2id[action_name]

        # 保存该视频的所有帧记录
        for frame_file in frame_list:
            frame_path = os.path.join(video_dir, frame_file)
            data.append({
                'Index': global_idx,
                'DataName': 'GynSurg_Action',
                'Year': 2022,  # 可补充正确年份
                'Case_Name': video_name,
                'Case_ID': case_id,
                'Frame_Path': frame_path,
                'Phase_GT': phase_gt,        # action_name映射到数字标签
                'Phase_Name': action_name,   # 动作类别作为阶段名称
                'Split': ''                  # 先空，后面分配
            })
            global_idx += 1

# 任务2:生成CSV并划分训练/验证/测试集（按动作类别分）
df = pd.DataFrame(data)
all_rows = []

for action_name in df['Phase_Name'].unique():
    df_action = df[df['Phase_Name'] == action_name].copy()
    video_names = df_action['Case_Name'].unique()
    # 以视频为单位分
    train_videos, testval_videos = train_test_split(video_names, test_size=0.3, random_state=42)
    val_videos, test_videos = train_test_split(testval_videos, test_size=0.5, random_state=42)

    df.loc[df['Case_Name'].isin(train_videos) & (df['Phase_Name']==action_name), 'Split'] = 'train'
    df.loc[df['Case_Name'].isin(val_videos) & (df['Phase_Name']==action_name), 'Split'] = 'val'
    df.loc[df['Case_Name'].isin(test_videos) & (df['Phase_Name']==action_name), 'Split'] = 'test'

# 重新排序Index
df = df.sort_values(by="Index").reset_index(drop=True)
df["Index"] = df.index

out_csv_train = "data/Surge_Frames/GynSurg_Action/train_metadata.csv"
out_csv_val = "data/Surge_Frames/GynSurg_Action/val_metadata.csv"
out_csv_test = "data/Surge_Frames/GynSurg_Action/test_metadata.csv"

df[df["Split"]=="train"].to_csv(out_csv_train, index=False)
df[df["Split"]=="val"].to_csv(out_csv_val, index=False)
df[df["Split"]=="test"].to_csv(out_csv_test, index=False)
print(f"Saved {len(df[df['Split']=='train'])} training frames to {out_csv_train}")
print(f"Saved {len(df[df['Split']=='val'])} validation frames to {out_csv_val}")
print(f"Saved {len(df[df['Split']=='test'])} test frames to {out_csv_test}")



Saved 9865 training frames to data/Surge_Frames/GynSurg_Action/train_metadata.csv
Saved 3321 validation frames to data/Surge_Frames/GynSurg_Action/val_metadata.csv
Saved 1835 test frames to data/Surge_Frames/GynSurg_Action/test_metadata.csv


In [1]:
## fix unlabeled csv

import pandas as pd

# csv_path = "data/Surge_Frames/EndoFM_Private/clips_64f/unlabeled_dense_64f_detailed.csv"
# csv_path = "data/Surge_Frames/EndoFM_Colonoscopic/clips_64f/unlabeled_dense_64f_detailed.csv"
# csv_path = "data/Surge_Frames/Private_SYSU_Brochiscopy_Cut/clips_64f/unlabeled_dense_64f_detailed.csv"
# csv_path = "data/Surge_Frames/Private_PWH_TSS_79/clips_64f/unlabeled_dense_64f_detailed.csv"
# csv_path = "data/Surge_Frames/Private_PUMCH/clips_64f/unlabeled_dense_64f_detailed.csv"

csv_path = "data/Surge_Frames/Private_HKU-SZH_Brain_Open_Surgery/clips_64f/unlabeled_dense_64f_detailed.csv"

df = pd.read_csv(csv_path)
df["label"] = -1
df["label_name"] = "none"
df.to_csv(csv_path, index=True, index_label="Index")



In [ ]:
import pandas as pd

# 1. 读取包含多余列的CSV文件
# 注意：如果CSV的第一行是列名，header=0（默认）；如果没有列名，需调整header参数
csv_path = "data/Surge_Frames/Private_PUMCH/clips_64f/unlabeled_dense_64f_detailed.csv"
csv_path = "data/Surge_Frames/Private_HKU-SZH_Brain_Open_Surgery/clips_64f/unlabeled_dense_64f_detailed.csv"
df = pd.read_csv(csv_path)

# 2. 删除第一列和第二列（列索引为0和1）
# 方式1：通过列索引删除（推荐，直接删除前两列）
df_cleaned = df.drop(df.columns[[0, 1]], axis=1)

# 3. 重新写入CSV，保留并命名索引为"Index"
df_cleaned.to_csv(csv_path, index=True, index_label="Index")

print("处理完成！已删除前两列并重新生成带正确索引的CSV。")

处理完成！已删除前两列并重新生成带正确索引的CSV。


In [4]:
 
## 在data/NeuroSurgery/pumch_fullvideo下，有若干个文件夹[2015_001  2017_001  2018_001  2019_001  2020_001  2021_001  2021_002  2021_003  2022_001],每个文件夹代表nian fen
### 每个年份的文件夹下，有若干 case的文件夹，比如[case0000_20151130140710  case0002_20151130105810  case0004_20140508121916  case0007_20140514081016  case0009_20151102112901  case0011_20151123111740]
#### 每个文件夹下有Video文件夹，Video文件夹下是concatenated_video.mp4

## 现在我想把年份和case的文件夹合并在一起，命名为{年份}_{case}，然后移动到data/NeuroSurgery/pumch_fullvideo_new下
### 把Video的文件夹干掉，只保留concatenated_video.mp4

import os
import shutil

# 定义源路径和目标路径
source_path = "data/NeuroSurgery/pumch_fullvideo"
target_path = "data/NeuroSurgery/pumch_fullvideo_new"

# 只遍历一层年份+case的父子关系（不是所有组合）
for year_folder in os.listdir(source_path):
    year_path = os.path.join(source_path, year_folder)
    if not os.path.isdir(year_path):
        continue
    # 只遍历这个年份下实际存在的case文件夹（不组合）
    for case_folder in os.listdir(year_path):
        case_path = os.path.join(year_path, case_folder)
        if not os.path.isdir(case_path):
            continue
        # 只关注case_path/Video/concatenated_video.mp4的情况
        mp4_path = os.path.join(case_path, "Video", "concatenated_video.mp4")
        if os.path.exists(mp4_path):
            # 确保目标目录存在
            os.makedirs(target_path, exist_ok=True)
            # 新文件名
            new_name = f"{year_folder}_{case_folder}.mp4"
            new_file_full = os.path.join(target_path, new_name)
            # 移动并重命名
            shutil.move(mp4_path, new_file_full)
            # 删除空的Video文件夹
            video_dir = os.path.join(case_path, "Video")
            try:
                os.rmdir(video_dir)
            except Exception:
                pass


In [ ]:
### clear tss

import os
import shutil

# -------------------------- 基础配置 --------------------------
root_folder = "/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo"
new_folder = "/public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos"
target_ext = ".mp4"

# -------------------------- 创建新文件夹 --------------------------
if not os.path.exists(new_folder):
    os.makedirs(new_folder)
    print(f"已创建新文件夹：{new_folder}")
else:
    print(f"新文件夹已存在：{new_folder}")

# -------------------------- 递归查找MP4文件 --------------------------
found_mp4_files = []
for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.lower().endswith(target_ext):
            full_file_path = os.path.join(root, file)
            found_mp4_files.append(full_file_path)

if not found_mp4_files:
    print(f"未找到任何 {target_ext} 文件")
else:
    print(f"共找到 {len(found_mp4_files)} 个 {target_ext} 文件，开始整理...")

# -------------------------- 按规则移动文件（处理空格和中文） --------------------------
for mp4_path in found_mp4_files:
    # 解析文件信息
    file_dir = os.path.dirname(mp4_path)  # 父文件夹路径
    file_name = os.path.basename(mp4_path)  # 原始文件名（含中文和空格）
    
    # 1. 处理前缀：去掉根目录，替换路径分隔符为"_", 空格替换为"-"
    prefix = file_dir.replace(root_folder, "").strip(os.sep)  # 去掉根目录前缀
    prefix = prefix.replace(os.sep, "_")  # 路径分隔符→下划线
    prefix = prefix.replace(" ", "-")  # 空格→横线
    
    # 2. 处理原始文件名：空格替换为"-"（保留中文）
    cleaned_file_name = file_name.replace(" ", "-")
    
    # 3. 生成新文件名
    if prefix:
        new_file_name = f"{prefix}_{cleaned_file_name}"
    else:
        new_file_name = cleaned_file_name  # 根目录下的文件直接用清理后的文件名
    
    # 4. 处理重复文件名
    new_file_path = os.path.join(new_folder, new_file_name)
    counter = 1
    while os.path.exists(new_file_path):
        name_without_ext, ext = os.path.splitext(new_file_name)
        new_file_path = os.path.join(new_folder, f"{name_without_ext}_{counter}{ext}")
        counter += 1
    
    # 执行移动
    shutil.move(mp4_path, new_file_path)
    print(f"已移动：{mp4_path} → {new_file_path}")

print(f"\n整理完成！文件保存至：{new_folder}")

In [2]:
import os
import shutil

# -------------------------- 基础配置 --------------------------
root_folder = "/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo"
new_folder = "/public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos"
target_ext = ".mp4"

# -------------------------- 创建新文件夹（若不存在） --------------------------
if not os.path.exists(new_folder):
    os.makedirs(new_folder)
    print(f"已创建新文件夹：{new_folder}")
else:
    print(f"新文件夹已存在：{new_folder}")

# -------------------------- 递归查找MP4文件 --------------------------
found_mp4_files = []
for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.lower().endswith(target_ext):
            full_file_path = os.path.join(root, file)
            found_mp4_files.append(full_file_path)

if not found_mp4_files:
    print(f"未找到任何 {target_ext} 文件，无需移动")
else:
    print(f"共找到 {len(found_mp4_files)} 个 {target_ext} 文件，开始移动...\n")

# -------------------------- 执行文件移动 --------------------------
success_count = 0
fail_count = 0
fail_files = []

for mp4_path in found_mp4_files:
    try:
        # 解析文件信息
        file_dir = os.path.dirname(mp4_path)
        file_name = os.path.basename(mp4_path)
        
        # 处理前缀（路径分隔符→下划线，空格→横线）
        prefix = file_dir.replace(root_folder, "").strip(os.sep)
        prefix = prefix.replace(os.sep, "_")
        prefix = prefix.replace(" ", "-")
        
        # 处理原始文件名（空格→横线）
        cleaned_file_name = file_name.replace(" ", "-")
        
        # 生成新文件名
        if prefix:
            new_file_name = f"{prefix}_{cleaned_file_name}"
        else:
            new_file_name = cleaned_file_name
        
        # 处理重复文件名
        new_file_path = os.path.join(new_folder, new_file_name)
        counter = 1
        while os.path.exists(new_file_path):
            name_without_ext, ext = os.path.splitext(new_file_name)
            new_file_path = os.path.join(new_folder, f"{name_without_ext}_{counter}{ext}")
            counter += 1
        
        # 执行移动
        shutil.move(mp4_path, new_file_path)
        print(f"成功：{mp4_path} → {new_file_path}")
        success_count += 1
    
    except Exception as e:
        print(f"失败：{mp4_path}（错误：{str(e)}）")
        fail_count += 1
        fail_files.append(mp4_path)

# -------------------------- 移动结果统计 --------------------------
print("\n" + "=" * 60)
print(f"移动完成！总计：{len(found_mp4_files)} 个文件")
print(f"成功移动：{success_count} 个")
print(f"移动失败：{fail_count} 个")

if fail_files:
    print("\n失败的文件列表：")
    for f in fail_files:
        print(f"- {f}")
print("=" * 60)
print(f"所有成功移动的文件已保存至：{new_folder}")

已创建新文件夹：/public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos
共找到 307 个 .mp4 文件，开始移动...

成功：/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo/2024-12-11 杨昌龙 脊索瘤/concatenated_video.mp4 → /public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos/2024-12-11-杨昌龙-脊索瘤_concatenated_video.mp4
成功：/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo/2025-02-11 林康利 巨大骨软骨瘤/concatenated_video.mp4 → /public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos/2025-02-11-林康利-巨大骨软骨瘤_concatenated_video.mp4
成功：/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo/2025-02-11 林康利 巨大骨软骨瘤/2025-02-11_220332/concatenated_video.mp4 → /public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Videos/2025-02-11-林康利-巨大骨软骨瘤_2025-02-11_220332_concatenated_video.mp4
成功：/public/datasets/neurosurg_videos/NeuroSurgery/TSS_fullvideo/2025-02-11 林康利 巨大骨软骨瘤/2025-02-11_224309/concatenated_video.mp4 → /public/datasets/neurosurg_videos/NeuroSurgery/Organized_TSS_Video

In [5]:
# 统计这些视频的平均时长 data/Landscopy/GynSurg_Action_Segments/*/*.mp4

import os
import pandas as pd
import subprocess
# 定义数据路径
data_path = "data/Landscopy/GynSurg_Action_Segments"

# 初始化列表来存储视频时长
video_durations = []

import glob

vids = glob.glob(data_path + "/*/*.mp4")

for vid in vids:
    command = f"ffprobe -v error -select_streams v:0 -show_entries stream=duration -of default=noprint_wrappers=1:nokey=1 {vid}"
    duration = subprocess.check_output(command, shell=True).decode('utf-8').strip()
    video_durations.append(float(duration))

# 计算平均时长
if video_durations:
    average_duration = sum(video_durations) / len(video_durations)



# 计算平均时长
if video_durations:
    average_duration = sum(video_durations) / len(video_durations)

    print(f"平均时长: {average_duration} 秒")
    print(f"平均时长: {average_duration/60} 分钟")
    print(f"平均时长: {average_duration/3600} 小时")    


平均时长: 25.969764985899282 秒
平均时长: 0.4328294164316547 分钟
平均时长: 0.007213823607194245 小时


In [2]:
# 把四个数据集的csv文件中Frame_Path里'Surge_Frames'前的所有内容换成'data'，实现路径为当前data下的相对路径

import pandas as pd
import os
import glob

data_list = [
"data/Surge_Frames/Atlas_labeled",
"data/Surge_Frames/Private_pumch_labeled",
"data/Surge_Frames/Private_pwh_labeled",
"data/Surge_Frames/Private_TSS_labeled"
]

# for d in data_list:
#     csv_files = glob.glob(os.path.join(d, "*.csv"))
#     for csv_fp in csv_files:
#         df = pd.read_csv(csv_fp)
#         if "Frame_Path" in df.columns:
#             # 只替换Surge_Frames之前的内容为"data"
#             def fix_path(path):
#                 idx = path.find("Surge_Frames")
#                 if idx != -1:
#                     return os.path.join("data", path[idx:])
#                 else:
#                     return path
#             df["Frame_Path"] = df["Frame_Path"].apply(fix_path)
#             # 覆盖保存
#             df.to_csv(csv_fp, index=False)

# 把这四个数据集的csv文件中的index改成Index

import pandas as pd
import os
import glob

data_list = [
    "data/Surge_Frames/Atlas_labeled",
    "data/Surge_Frames/Private_pumch_labeled",
    "data/Surge_Frames/Private_pwh_labeled",
    "data/Surge_Frames/Private_TSS_labeled"
]

for d in data_list:
    csv_files = glob.glob(os.path.join(d, "*.csv"))
    for csv_fp in csv_files:
        df = pd.read_csv(csv_fp)
        if "index" in df.columns:
            df = df.rename(columns={"index": "Index"})
            df.to_csv(csv_fp, index=False)




